In [1]:
import sqlite3
from datetime import datetime

DB = 'lab3_blockchain.db'

def conn():
    c = sqlite3.connect(DB, timeout=5)
    c.execute("PRAGMA journal_mode=WAL")
    c.row_factory = sqlite3.Row
    return c

In [2]:
with conn() as c:
    c.execute("""
        CREATE TABLE IF NOT EXISTS event_stream (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            type TEXT,
            entity_id TEXT,
            processed BOOLEAN DEFAULT 0,
            created_at TEXT DEFAULT CURRENT_TIMESTAMP
        )
    """)

In [3]:
def add_block(id, view, desc):
    with conn() as c:
        c.execute("INSERT OR IGNORE INTO BLOCKS (id, view, desc) VALUES (?,?,?)", (id, view, desc))
        c.execute("INSERT INTO event_stream (type, entity_id) VALUES (?,?)", ('block', id))

def add_vote(block_id, voter_id, timestamp, source_id):
    with conn() as c:
        c.execute("INSERT OR IGNORE INTO VOTES (block_id, voter_id, timestamp, source_id) VALUES (?,?,?,?)",
                  (block_id, voter_id, timestamp, source_id))
        c.execute("INSERT INTO event_stream (type, entity_id) VALUES (?,?)",
                  ('vote', f"{block_id}:{voter_id}"))

def add_person(id, name, addr):
    with conn() as c:
        c.execute("INSERT OR IGNORE INTO PERSONS (id, name, addr) VALUES (?,?,?)", (id, name, addr))
        c.execute("INSERT INTO event_stream (type, entity_id) VALUES (?,?)", ('person', str(id)))

def add_source(id, ip, country):
    with conn() as c:
        c.execute("INSERT OR IGNORE INTO SOURCES (id, ip_addr, country_code) VALUES (?,?,?)", (id, ip, country))
        c.execute("INSERT INTO event_stream (type, entity_id) VALUES (?,?)", ('source', str(id)))

In [4]:
def process_one_event(event):
    typ = event['type']
    eid = event['entity_id']
    if typ == 'block':
        with conn() as c:
            row = c.execute("SELECT * FROM BLOCKS WHERE id=?", (eid,)).fetchone()
            print(f"Обрабка блоку {row['id']}: {row['desc']}")
    elif typ == 'vote':
        bid, vid = eid.split(':')
        with conn() as c:
            row = c.execute("SELECT * FROM VOTES WHERE block_id=? AND voter_id=?", (bid, vid)).fetchone()
            print(f"Голос за блок {row['block_id']} від {row['voter_id']}")
    elif typ == 'person':
        with conn() as c:
            row = c.execute("SELECT * FROM PERSONS WHERE id=?", (eid,)).fetchone()
            print(f"Особа {row['id']}: {row['name']}")
    elif typ == 'source':
        with conn() as c:
            row = c.execute("SELECT * FROM SOURCES WHERE id=?", (eid,)).fetchone()
            print(f"Джерело {row['id']}: {row['ip_addr']}")

def process_all():
    with conn() as c:
        events = c.execute("SELECT id, type, entity_id FROM event_stream WHERE processed=0").fetchall()
    for ev in events:
        process_one_event(ev)
        with conn() as c:
            c.execute("UPDATE event_stream SET processed=1 WHERE id=?", (ev['id'],))

In [5]:
add_block('0xabc', 100, 'Перший блок')
add_vote('0xabc', 1, '2025-03-23 12:00', 2)
add_person(1, 'Хуй Івана', 'Львів')
add_source(2, '192.168.1.1', 'UA')
process_all()

Обрабка блоку 0xabc: Перший блок
Голос за блок 0xabc від 1
Особа 1: Катерина Слободяник
Джерело 2: 198.51.100.2
